# Data Cleaning in Pandas and NumPy

In the real world, data is rarely clean. It comes with missing values, duplicate records, incorrect data types, formatting inconsistencies, and extreme outliers. 

As the saying goes: *"Data Scientists spend 80% of their time cleaning data, and 20% of their time complaining about cleaning data."* Having clean data is absolutely critical because the performance of any analysis or Machine Learning model directly depends on the quality of its inputs (**"Garbage In, Garbage Out"**).

---

## 1. Handling Missing Data (NaNs)

Pandas uses `NaN` (Not a Number) from NumPy to represent missing numerical and object data.

### Identifying Missing Values
To find missing values, use `.isna()` or `.isnull()` (they are identical):




In [27]:
import pandas as pd
import numpy as np

# Comprehensive sample uncleaned DataFrame (df) for demonstration cells
data = {
    'FirstName': ['Alice', 'Bob', 'Charlie', 'David', 'Eva', 'Alice', 'Frank'],
    'LastName': ['Smith', 'Jones', 'Brown', 'Davis', 'Miller', 'Smith', 'Wilson'],
    'Name': ['Alice Smith', 'Bob Jones', 'Charlie Brown', 'David Davis', 'Eva Miller', 'Alice Smith', 'Frank Wilson'],
    'Email': ['alice@email.com', np.nan, 'charlie@email.com', 'david@email.com', np.nan, 'alice@email.com', 'frank@email.com'],
    'Age': ['25', '30', '35', np.nan, '45', '25', '28'],
    'Salary': [50000.0, 60000.0, np.nan, 75000.0, 80000.0, 50000.0, 250000.0],
    'Department': ['IT', 'HR', 'Sales', 'IT', np.nan, 'IT', 'Sales'],
    'Status': ['Active', np.nan, 'Active', 'Pending', 'Active', 'Active', np.nan],
    'StockPrice': [150.0, 152.0, np.nan, np.nan, 155.0, 150.0, 158.0],
    'Price': [' $100.00 ', ' $200.50 ', ' $150.00 ', ' invalid ', ' $300.00 ', ' $100.00 ', ' $250.00 '],
    'JoinDate': ['2023-01-15', '2022/05/20', '2024-03-10', 'invalid_date', '2021-11-01', '2023-01-15', '2020-08-19'],
    'YearsOfExperience': [2, 5, 8, 10, 45, 2, 6],
    'Country': ['USA', 'UK', 'Canada', 'USA', 'Germany', 'USA', 'UK'],
    'Score': [85.0, 90.0, 78.0, np.nan, 92.0, 85.0, 88.0]
}
df = pd.DataFrame(data)
print('Sample uncleaned DataFrame (df) initialized:')
print(df)


Sample uncleaned DataFrame (df) initialized:
  FirstName LastName           Name              Email  Age    Salary  \
0     Alice    Smith    Alice Smith    alice@email.com   25   50000.0   
1       Bob    Jones      Bob Jones                NaN   30   60000.0   
2   Charlie    Brown  Charlie Brown  charlie@email.com   35       NaN   
3     David    Davis    David Davis    david@email.com  NaN   75000.0   
4       Eva   Miller     Eva Miller                NaN   45   80000.0   
5     Alice    Smith    Alice Smith    alice@email.com   25   50000.0   
6     Frank   Wilson   Frank Wilson    frank@email.com   28  250000.0   

  Department   Status  StockPrice      Price      JoinDate  YearsOfExperience  \
0         IT   Active       150.0   $100.00     2023-01-15                  2   
1         HR      NaN       152.0   $200.50     2022/05/20                  5   
2      Sales   Active         NaN   $150.00     2024-03-10                  8   
3         IT  Pending         NaN   invalid   



### Strategy A: Dropping Missing Values (`dropna`)
If a column or row has too many missing values, or if those rows are useless without that data, you can drop them:




In [29]:
# Drop any row containing at least one missing value
df_clean = df.dropna()

# Drop rows ONLY if specific columns have missing values
df_clean = df.dropna(subset=["Email", "Salary"])

# Drop columns containing at least one missing value
df_clean = df.dropna(axis=1)

# Drop rows only if ALL values in the row are missing
df_clean = df.dropna(how="all")




### Strategy B: Filling Missing Values (`fillna`)
Often, dropping data loses valuable information. Instead, we can impute (fill) missing values:




In [30]:
# Fill all missing values in the DataFrame with a constant
df_filled = df.fillna(0)

# Fill a specific column with a constant value
df["Status"] = df["Status"].fillna("Unknown")

# Impute with summary statistics (Mean, Median, or Mode)
salary_median = df["Salary"].median()
df["Salary"] = df["Salary"].fillna(salary_median)

# Forward fill (propagate last valid observation forward)
df["StockPrice"] = df["StockPrice"].ffill()

# Backward fill (propagate next valid observation backward)
df["StockPrice"] = df["StockPrice"].bfill()




---

## 2. Dealing with Duplicates

Duplicate records often creep in during data aggregation or database scraping.




In [31]:
# Check which rows are duplicate (returns True for subsequent duplicates)
print(df.duplicated())

# Count total duplicate rows
print(df.duplicated().sum())

# Check duplicates based on a subset of columns
print(df.duplicated(subset=["FirstName", "LastName"]).sum())

# Drop duplicate rows (keeping the first occurrence)
df_unique = df.drop_duplicates()

# Drop duplicates based on specific columns and keep the last occurrence
df_unique = df.drop_duplicates(subset=["Email"], keep="last")


0    False
1    False
2    False
3    False
4    False
5     True
6    False
dtype: bool
1
1




---

## 3. Correcting Faulty Data Types

Sometimes numeric columns are loaded as text because they contain characters like currency symbols, commas, or typos.

### Checking Column Types



In [32]:
print(df.dtypes)
# or
df.info()


FirstName                str
LastName                 str
Name                     str
Email                    str
Age                      str
Salary               float64
Department               str
Status                   str
StockPrice           float64
Price                    str
JoinDate                 str
YearsOfExperience      int64
Country                  str
Score                float64
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   FirstName          7 non-null      str    
 1   LastName           7 non-null      str    
 2   Name               7 non-null      str    
 3   Email              5 non-null      str    
 4   Age                6 non-null      str    
 5   Salary             7 non-null      float64
 6   Department         6 non-null      str    
 7   Status             7 non-null      str    
 8   Stock



### Basic Casting (`astype`)
If the column is completely clean but just stored as the wrong type, cast it directly:




In [33]:
# Cast integer column to float
df["Age"] = df["Age"].astype(float)

# Cast string column to category (saves memory for repeated text)
df["Department"] = df["Department"].astype("category")




### Messy Data Casting (`pd.to_numeric` & `pd.to_datetime`)
If a numeric column contains text or invalid characters, `.astype(float)` will crash. Instead, use Pandas converter functions with error handling:




In [34]:
# errors="coerce" replaces invalid parsing inputs with NaN
df["Salary"] = pd.to_numeric(df["Salary"], errors="coerce")

# Now fill the newly created NaNs with the median
df["Salary"] = df["Salary"].fillna(df["Salary"].median())




For date values, convert them to standard datetime format so you can extract date parts (year, month, weekday) easily:




In [35]:
# Convert a string column to standard pandas Datetime objects
df["JoinDate"] = pd.to_datetime(df["JoinDate"], errors="coerce")

# Extract the year, month, or day of the week
df["JoinYear"] = df["JoinDate"].dt.year
df["JoinDayName"] = df["JoinDate"].dt.day_name()




---

## 4. String Cleaning (The `.str` Accessor)

When working with textual data, Pandas provides a powerful `.str` accessor that applies string methods element-wise to an entire Series:




In [36]:
# 1. Strip whitespace
df["Email"] = df["Email"].str.strip()

# 2. Case standardization
df["Name"] = df["Name"].str.title()
df["Country"] = df["Country"].str.upper()

# 3. Clean currency strings (e.g. "$1,200.50" -> 1200.50)
df["Price"] = df["Price"].str.replace("$", "", regex=False)
df["Price"] = df["Price"].str.replace(",", "", regex=False)
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

# 4. Check if string contains a pattern (Boolean indexing)
edu_emails = df[df["Email"].str.contains(r"\.edu$", na=False, regex=True)]




---

## 5. Detecting and Handling Outliers

Outliers are extreme data points that deviate significantly from the rest of the dataset. While some outliers are legitimate, others are errors (e.g. age entered as `999`).

### The Interquartile Range (IQR) Method
The IQR method is a classic statistical tool to detect outliers:

1. Calculate the first quartile ($Q_1$ or 25th percentile).
2. Calculate the third quartile ($Q_3$ or 75th percentile).
3. Find the Interquartile Range: $IQR = Q_3 - Q_1$.
4. Define the bounds:
   - **Lower Bound** = $Q_1 - (1.5 \times IQR)$
   - **Upper Bound** = $Q_3 + (1.5 \times IQR)$
5. Points outside this range are considered outliers.




In [37]:
# Compute Q1 and Q3
Q1 = df["Score"].quantile(0.25)
Q3 = df["Score"].quantile(0.75)
IQR = Q3 - Q1

# Define boundaries
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identify outliers
outliers = df[(df["Score"] < lower_bound) | (df["Score"] > upper_bound)]
print(f"Detected {len(outliers)} outliers")

# Filter out outliers (keep values inside boundaries)
df_no_outliers = df[(df["Score"] >= lower_bound) & (df["Score"] <= upper_bound)]


Detected 1 outliers




---

## 6. Best Practices in Data Cleaning

1. **Keep a Copy of Your Raw Data**: Always load raw data and create a copy (`df_clean = df.copy()`) before applying transformations.
2. **Inspect After Every Step**: Run `df.shape` or `df.isna().sum()` after dropping rows or columns to verify the effect.
3. **Avoid SettingWithCopyWarning**: Do not modify subsets of DataFrames directly (e.g. `df[df.Age > 30]["Status"] = "Active"`). Instead, use `.loc` (e.g. `df.loc[df.Age > 30, "Status"] = "Active"`) or `.copy()`.
4. **Use Vectorized Operations**: Avoid row-by-row iteration (`for index, row in df.iterrows()`). Pandas methods are implemented in C and run orders of magnitude faster.

## Resources

- **Official Pandas Documentation** – https://pandas.pydata.org/docs/
- **Official NumPy Documentation** – https://numpy.org/doc/
- **Real Python: Data Cleaning with Pandas** – https://realpython.com/pandas-data-cleaning/
- **Kaggle: Data Cleaning Cheat Sheet** – https://www.kaggle.com/learn/data-cleaning-cheat-sheet
- **Python Data Cleaning Cookbook (O'Reilly)** – https://www.oreilly.com/library/view/python-data-cleaning/9781492048801/
- **Effective Pandas: Tips and Tricks** – https://towardsdatascience.com/effective-pandas-tips-tricks-75f0e55310c9



# Examples & Demonstrations
Below are runnable examples demonstrating the concepts covered in this module.


In [38]:
# Examples: Data Cleaning in Pandas and NumPy

import pandas as pd
import numpy as np
import io

# Simulated messy CSV dataset containing multiple issues:
# - NaNs (missing age and salary)
# - Duplicated entries (exact duplicate for 'Sara', subset duplicate for 'Omar')
# - Faulty types (Salary is string with currency characters and commas)
# - Faulty date formatting
# - Outliers (An extremely high value in YearsOfExperience, perhaps a typo)
messy_csv_data = """Name,Age,Department,Salary,YearsOfExperience,JoinDate
Hamza,25,IT,"$95,000",3,2023-01-15
Ali,,Marketing,"$62,000",1,2024-06-01
Sara,30,HR,"$75,000",6,2020-11-10
Sara,30,HR,"$75,000",6,2020-11-10
Hiba,28,IT,,5,2021/05/20
Omar,35,Sales,"$85,000",10,2018-09-12
Omar,35,Sales,"$85,000",99,2018-09-12
Noor,24,Marketing,"$64,000",2,invalid_date
"""


In [39]:
def detect_and_handle_nans(df):
    print("\n--- 1. Detecting and Handling Missing Values (NaNs) ---")
    
    # Check for missing values
    print("Missing values per column:")
    print(df.isna().sum())
    
    # Create a copy to clean
    df_clean = df.copy()
    
    # Strategy A: Impute numerical missing values with Median
    # Note: Salary needs to be numeric first, but let's clean Age first
    age_median = df_clean["Age"].median()
    print(f"\nImputing missing 'Age' values with median: {age_median}")
    df_clean["Age"] = df_clean["Age"].fillna(age_median)
    
    print("\nDataFrame after filling 'Age':")
    print(df_clean[["Name", "Age", "Department"]])
    
    return df_clean


In [40]:
def remove_duplicates(df):
    print("\n--- 2. Removing Duplicate Records ---")
    
    print("Total exact duplicate rows:", df.duplicated().sum())
    
    # Drop exact duplicates
    df_clean = df.drop_duplicates()
    print("Exact duplicates dropped. Row count went from", len(df), "to", len(df_clean))
    
    # Drop duplicates on a specific subset (e.g., Name + Department)
    # This addresses Omar who has one correct row and one faulty row (experience = 99)
    # We keep the first occurrence
    df_clean = df_clean.drop_duplicates(subset=["Name", "Department"], keep="first")
    print("Subset duplicates (Name + Department) dropped. Current rows:", len(df_clean))
    print(df_clean[["Name", "Department", "YearsOfExperience"]])
    
    return df_clean


In [41]:
def parse_and_cast_types(df):
    print("\n--- 3. Type Conversion and String Cleaning ---")
    
    df_clean = df.copy()
    
    # 1. Clean Salary String: remove '$' and ',' and convert to numeric
    print("Original 'Salary' values and types:")
    print(df_clean["Salary"])
    
    df_clean["Salary"] = df_clean["Salary"].astype(str) # ensure it is string
    df_clean["Salary"] = df_clean["Salary"].str.replace("$", "", regex=False)
    df_clean["Salary"] = df_clean["Salary"].str.replace(",", "", regex=False)
    
    # Convert to numeric, turn bad strings (like 'nan' or empty string) into NaN
    df_clean["Salary"] = pd.to_numeric(df_clean["Salary"], errors="coerce")
    
    # Fill missing salaries with average salary
    avg_salary = df_clean["Salary"].mean()
    df_clean["Salary"] = df_clean["Salary"].fillna(avg_salary)
    
    print("\nCleaned 'Salary' (converted to float and filled NaN):")
    print(df_clean[["Name", "Salary"]])
    
    # 2. Parse Datetime: parse Dates, coerce bad formatting to NaNs
    print("\nParsing 'JoinDate' column:")
    df_clean["JoinDate"] = pd.to_datetime(df_clean["JoinDate"], errors="coerce")
    print(df_clean[["Name", "JoinDate"]])
    
    # Check what data type JoinDate is now
    print("\nData types after conversion:")
    print(df_clean.dtypes)
    
    return df_clean


In [42]:
def filter_outliers_iqr(df):
    print("\n--- 4. Detecting and Handling Outliers ---")
    
    df_clean = df.copy()
    
    # Let's inspect YearsOfExperience
    print("YearsOfExperience column:")
    print(df_clean[["Name", "YearsOfExperience"]])
    
    # IQR Method calculation
    Q1 = df_clean["YearsOfExperience"].quantile(0.25)
    Q3 = df_clean["YearsOfExperience"].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    print(f"\nIQR statistics: Q1={Q1}, Q3={Q3}, IQR={IQR}")
    print(f"Valid range for YearsOfExperience: [{lower_bound}, {upper_bound}]")
    
    # Identify outliers
    outliers = df_clean[
        (df_clean["YearsOfExperience"] < lower_bound) | 
        (df_clean["YearsOfExperience"] > upper_bound)
    ]
    print("\nOutliers detected:")
    print(outliers[["Name", "YearsOfExperience"]])
    
    # Filter out outliers (keep valid rows)
    df_filtered = df_clean[
        (df_clean["YearsOfExperience"] >= lower_bound) & 
        (df_clean["YearsOfExperience"] <= upper_bound)
    ]
    
    print("\nDataFrame after removing outliers:")
    print(df_filtered[["Name", "YearsOfExperience"]])
    
    return df_filtered


In [43]:
print("=== RAW UNCLEANED DATA ===")
df_raw = pd.read_csv(io.StringIO(messy_csv_data))
print(df_raw)
print("=" * 30)
df_step1 = detect_and_handle_nans(df_raw)
df_step2 = remove_duplicates(df_step1)
df_step3 = parse_and_cast_types(df_step2)
df_final = filter_outliers_iqr(df_step3)
print("\n=== FINAL CLEANED DATASET ===")
print(df_final)
print("=" * 30)


=== RAW UNCLEANED DATA ===
    Name   Age Department   Salary  YearsOfExperience      JoinDate
0  Hamza  25.0         IT  $95,000                  3    2023-01-15
1    Ali   NaN  Marketing  $62,000                  1    2024-06-01
2   Sara  30.0         HR  $75,000                  6    2020-11-10
3   Sara  30.0         HR  $75,000                  6    2020-11-10
4   Hiba  28.0         IT      NaN                  5    2021/05/20
5   Omar  35.0      Sales  $85,000                 10    2018-09-12
6   Omar  35.0      Sales  $85,000                 99    2018-09-12
7   Noor  24.0  Marketing  $64,000                  2  invalid_date

--- 1. Detecting and Handling Missing Values (NaNs) ---
Missing values per column:
Name                 0
Age                  1
Department           0
Salary               1
YearsOfExperience    0
JoinDate             0
dtype: int64

Imputing missing 'Age' values with median: 30.0

DataFrame after filling 'Age':
    Name   Age Department
0  Hamza  25.0     

# Practice Exercises
EXERCISES: The Data Cleaning Guru

This script contains 3 practical exercises on data cleaning.
Complete the TODO sections to solve them.

Ensure you are using Pandas vectorized methods instead of loops!

In [44]:
import pandas as pd
import numpy as np

# =====================================================================
# EXERCISE 1: Null Value Detective
# =====================================================================
# We have a dictionary of messy climate sensor readings.
# Some values are missing due to sensor malfunctions.
sensor_data = {
    "SensorID": ["S01", "S02", "S03", "S04", "S05", "S06", "S07"],
    "Temperature": [22.5, np.nan, 24.1, 19.8, np.nan, 21.3, 23.0],
    "Humidity": [45, 50, np.nan, 39, np.nan, 42, 48],
    "Location": ["Room A", "Room B", "Room A", np.nan, "Room C", "Room B", "Room A"]
}

print("=== Exercise 1: Original Sensor Data ===")
df_sensors = pd.DataFrame(sensor_data)
print(df_sensors)
print("-" * 40)

# TODO 1: Print the number of missing values (NaN) in each column
print("TODO 1: Missing values count:")
missing_counts = df_sensors.isna().sum()
print(missing_counts)
print("-" * 40)

# TODO 2: Clean the Temperature column by dropping any row where Temperature is missing
print("TODO 2: Dropping rows with missing Temperature:")
df_sensors_temp_clean = df_sensors.dropna(subset=["Temperature"]).copy()
print(df_sensors_temp_clean)
print("-" * 40)

# TODO 3: Clean the Humidity column by filling missing values with the average Humidity of the dataset
print("TODO 3: Imputing Humidity with Mean:")
mean_humidity = df_sensors_temp_clean["Humidity"].mean()
df_sensors_temp_clean["Humidity"] = df_sensors_temp_clean["Humidity"].fillna(mean_humidity)
print(df_sensors_temp_clean)
print("-" * 40)

# TODO 4: Clean the Location column by filling missing values with the default location "Unknown Room"
print("TODO 4: Imputing Location with 'Unknown Room':")
df_sensors_temp_clean["Location"] = df_sensors_temp_clean["Location"].fillna("Unknown Room")
print(df_sensors_temp_clean)
print("=" * 60)


# =====================================================================
# EXERCISE 2: Deduplication Specialist
# =====================================================================
# We have a list of server access logs. Some lines are repeated
# due to network retries, and we need to keep only unique transactions.
log_data = {
    "RequestID": ["REQ101", "REQ102", "REQ101", "REQ103", "REQ104", "REQ103", "REQ103"],
    "User": ["Hamza", "Ali", "Hamza", "Sara", "Hiba", "Sara", "Sara"],
    "Endpoint": ["/home", "/profile", "/home", "/dashboard", "/settings", "/dashboard", "/dashboard"],
    "Status": [200, 404, 200, 200, 500, 200, 200]
}

print("\n=== Exercise 2: Original Log Data ===")
df_logs = pd.DataFrame(log_data)
print(df_logs)
print("-" * 40)

# TODO 1: Print the total number of exact duplicate rows in the log dataset
print("TODO 1: Exact duplicate rows count:")
duplicate_count = df_logs.duplicated().sum()
print(duplicate_count)
print("-" * 40)

# TODO 2: Drop exact duplicate rows and print the resulting DataFrame
print("TODO 2: DataFrame after dropping exact duplicates:")
df_logs_unique = df_logs.drop_duplicates()
print(df_logs_unique)
print("-" * 40)

# TODO 3: Drop duplicates based on the "RequestID" column and keep only the *last* transaction
print("TODO 3: Keeping the last occurrence of duplicate RequestIDs:")
df_logs_last = df_logs.drop_duplicates(subset=["RequestID"], keep="last")
print(df_logs_last)
print("=" * 60)


# =====================================================================
# EXERCISE 3: Financial Sanitizer (Currency & Type Cleaning)
# =====================================================================
# Below is a list of sales records with currency signs, commas,
# varying date formats, and some invalid numeric records.
sales_data = {
    "OrderID": ["TXN_01", "TXN_02", "TXN_03", "TXN_04", "TXN_05"],
    "Item": ["MacBook Pro", "iPhone 15 Pro", "AirPods Max", "USB-C Cable", "iPad Air"],
    "Price": [" $1,999.99 ", " $999.00 ", " $549.00 ", " $19.99 ", " invalid_price "],
    "TransactionDate": ["2024-01-10", "2024/02/15", "2023-12-05", "2024-03-01", "not_a_date"]
}

print("\n=== Exercise 3: Original Sales Data ===")
df_sales = pd.DataFrame(sales_data)
print(df_sales)
print("-" * 40)

# TODO 1: Clean the "Price" column:
#   - Strip leading/trailing whitespaces
#   - Remove "$" and "," characters
#   - Convert the column to numeric type (float) using pd.to_numeric (with errors="coerce")
#   - Fill any resulting NaN value in Price with the median price of the valid records
print("TODO 1: Clean and Convert 'Price' Column:")
df_sales["Price"] = df_sales["Price"].str.strip()
df_sales["Price"] = df_sales["Price"].str.replace("$", "", regex=False)
df_sales["Price"] = df_sales["Price"].str.replace(",", "", regex=False)
df_sales["Price"] = pd.to_numeric(df_sales["Price"], errors="coerce")

median_price = df_sales["Price"].median()
df_sales["Price"] = df_sales["Price"].fillna(median_price)

print(df_sales)
print("-" * 40)

# TODO 2: Convert "TransactionDate" to a Pandas datetime format, coercing errors to NaNs
# Hint: In modern Pandas, if date formats are mixed (e.g. some with "-" and some with "/"), 
# use format="mixed" to allow Pandas to parse each format individually.
print("TODO 2: Parse 'TransactionDate' as datetime:")
df_sales["TransactionDate"] = pd.to_datetime(df_sales["TransactionDate"], errors="coerce", format="mixed")
print(df_sales)
print("-" * 40)

# TODO 3: Filter the DataFrame to only show rows where TransactionDate is in the year 2024
# Hint: use the .dt accessor on datetime columns (e.g. df["Col"].dt.year == 2024)
print("TODO 3: Filter transactions in 2024:")
df_2024 = df_sales[df_sales["TransactionDate"].dt.year == 2024]
print(df_2024)
print("=" * 60)

if __name__ == "__main__":
    print("\nAll exercises prepared and completed successfully!")


=== Exercise 1: Original Sensor Data ===
  SensorID  Temperature  Humidity Location
0      S01         22.5      45.0   Room A
1      S02          NaN      50.0   Room B
2      S03         24.1       NaN   Room A
3      S04         19.8      39.0      NaN
4      S05          NaN       NaN   Room C
5      S06         21.3      42.0   Room B
6      S07         23.0      48.0   Room A
----------------------------------------
TODO 1: Missing values count:
SensorID       0
Temperature    2
Humidity       2
Location       1
dtype: int64
----------------------------------------
TODO 2: Dropping rows with missing Temperature:
  SensorID  Temperature  Humidity Location
0      S01         22.5      45.0   Room A
2      S03         24.1       NaN   Room A
3      S04         19.8      39.0      NaN
5      S06         21.3      42.0   Room B
6      S07         23.0      48.0   Room A
----------------------------------------
TODO 3: Imputing Humidity with Mean:
  SensorID  Temperature  Humidity Loca